# Clase 9 — Usar modelos que ya existen, sin creerles

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Bajar un modelo que entrenó otro y aplicarlo a tu
> corpus son tres líneas y cuarenta segundos. Saber si lo que te devolvió sirve para
> tu pregunta es el 70 % de esta clase, y todo tu trabajo profesional.

---

**Cuando termines este notebook vas a poder:**

1. Convertir cualquier corpus de texto en vectores y buscar en él **por significado**,
   no por palabra clave.
2. Elegir entre tres estrategias de clasificación **según cuántas etiquetas podés
   producir**, y defender la elección con una curva medida.
3. Reportar precisión, recall, F1 y matriz de confusión **diciendo en castellano llano
   qué significa cada uno** en tu problema, y compararlos contra un piso.
4. Detectar que tu clasificador está aprendiendo un atajo y no tu categoría.

**Lo que hace falta traer:** la clase 4 (partir los datos sin engañarse) y la clase 6
(qué es un embedding). Nada más.

---

## ⚠️ Leé esto antes de correr nada: los dos caminos

Este notebook tiene **dos caminos** y elige solo:

| | **Camino A — con Hugging Face** | **Camino B — sin red** |
|---|---|---|
| Necesita | internet y `transformers` | nada |
| Los vectores salen de | un modelo pre-entrenado en castellano | coocurrencia → PPMI → SVD, con numpy, sobre este mismo corpus |
| Cómo anda (medido) | con pocas etiquetas, mejor; con todas, parecido (F1 0,65 en los dos, en la tarea de hoy) | anda, y alcanza para todo lo que se enseña acá |

**El código de la evaluación —lo que importa de esta clase— es el mismo en los dos
caminos.** Cambia de dónde sale la matriz de vectores; no cambia ni una línea de lo que
viene después. Los números sí cambian un poco: los de las slides son los del camino B.

> **Aviso honesto de quien armó este material.** El notebook se construyó en un
> contenedor **sin acceso a `huggingface.co`**: ahí se corrió y se verificó entero el
> camino B. El camino A se corrió por primera vez en la revisión del 28-09-2026, en una
> máquina con red: bajó los modelos y terminó sin errores (los números están en
> `notas-docente.md` §10). Las celdas del camino A están marcadas con `# ── CAMINO A ──`,
> **fijan la revisión exacta de cada modelo** (`revision=`: la lección de la clase 10
> sobre citar con versión) y, si la descarga falla a mitad de camino, avisan y caen
> solas al camino B.
> Ojo: el camino A baja casi 1 GB. Con treinta personas en el mismo wifi, eso pesa: si
> la red del aula no da, poné `FORZAR_CAMINO_B = True` en la sección 1.

**Cómo se usa este notebook.** Corré las celdas en orden (`Shift + Enter`). Donde diga
`# TODO` te toca a vos.

**Las tres visualizaciones de esta clase** están en la carpeta `viz/` y abren en el
navegador **sin internet** (doble clic y listo). El notebook da por sentado que ya
jugaste con ellas:

| archivo | qué te muestra |
|---|---|
| `viz/zero-shot-vs-finetune.html` | cuántas etiquetas hacen falta, y cuál estrategia conviene con cuántas |
| `viz/similitud-semantica.html` | buscar por significado contra buscar por palabra, con sus falsos positivos |
| `viz/matriz-de-confusion.html` | **la central**: movés el umbral y la prevalencia, y mirás qué le pasa a cada métrica |

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el repo y
deja todo a mano; en tu máquina no hace prácticamente nada. Corrémela y seguí de largo.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports y semilla

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu resultado
reproducible, que es condición para poder afirmar algo — y es lo primero que un revisor
te va a pedir (clase 10).

In [ ]:
import re
import time
from collections import Counter

import numpy as np

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()

---

## 1. ¿Tenemos Hugging Face o no?

Antes de escribir una línea de análisis, averiguamos en qué camino estamos. La celda de
abajo **no rompe nunca**: prueba, y si no puede, lo dice y sigue.

Fijate en el patrón, porque es el que tienen que tener todos tus notebooks:
`intentar → si falla, avisar y usar el plan B`. Un notebook que explota porque se cayó
el wifi es un notebook que no se puede dar en un aula.

In [ ]:
# ¿Está la biblioteca? ¿Y hay red para bajar el modelo?
FORZAR_CAMINO_B = False     # True = ni lo intentes (el camino A baja casi 1 GB)

# Los modelos del camino A, con su revisión exacta: el hash del commit en el Hub.
# Citar un modelo sin revisión es como citar un libro sin edición (clase 10).
REVISION = {
    "pysentimiento/robertuito-sentiment-analysis": "a2cc0f67ebd705c55191e25a05ba23d885fcc09b",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
}

HAY_HF = False
MOTIVO_SIN_HF = ""

if FORZAR_CAMINO_B:
    tiene_biblioteca = False
    MOTIVO_SIN_HF = "lo pediste con FORZAR_CAMINO_B = True"
else:
    try:
        import transformers          # noqa: F401
        tiene_biblioteca = True
    except ImportError:
        tiene_biblioteca = False
        MOTIVO_SIN_HF = "la biblioteca `transformers` no está instalada"

if tiene_biblioteca:
    # No alcanza con que esté la biblioteca: hace falta llegar al servidor.
    import urllib.request
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=8)
        HAY_HF = True
    except Exception as e:
        MOTIVO_SIN_HF = f"no llego a huggingface.co ({type(e).__name__})"

if HAY_HF:
    print("CAMINO A: Hugging Face disponible. Vamos a bajar modelos de verdad.")
    print("   (Si una descarga falla más adelante, esa celda avisa y cae al camino B.)")
else:
    print("CAMINO B (sin red): vamos con los embeddings propios del curso.")
    print(f"   Motivo: {MOTIVO_SIN_HF}.")
    print("   No es un problema: todo lo que importa de esta clase corre igual.")
    if not FORZAR_CAMINO_B:
        print("   Si querés el camino A en Colab:  !pip install -q transformers")

---

## 2. Los datos, y **el piso antes que cualquier modelo**

Trabajamos con el corpus de prensa del curso: **1471 notas reales en castellano**, con
titular, tema, medio y una etiqueta verdadero/falso. Es el *Spanish Fake News Corpus 2.0*
(CC BY 4.0), y son **dos recolecciones pegadas**: 971 notas de la de enero a julio de
2018 y 500 de la de noviembre de 2020 a marzo de 2021 (ahí están todas las de Covid-19).
La procedencia completa está en `datos/README.md` §1.3; la época vuelve en la sección 11.

**Nuestra tarea principal va a ser una que hacés todo el tiempo aunque no la llames
así: armar un subcorpus temático.** En concreto: *¿esta nota es sobre política?*

Y lo primero que hacemos, **antes de tocar un modelo**, es calcular el piso: cuánto
acierta un clasificador que no mira nada y dice siempre la clase más común. Si tu modelo
no le gana a eso, no tenés un modelo: tenés un adorno caro.

In [ ]:
notas = datos.titulares_prensa(con_texto=True)
titulares = [n["titular"] for n in notas]

print(f"{len(notas)} notas de prensa (datos REALES, CC BY 4.0)")
print("\nTemas del corpus:")
for tema, cuantas in Counter(n["tema"] for n in notas).most_common():
    print(f"   {tema:<16} {cuantas:>5}")

# La etiqueta de nuestra tarea: 1 = la nota es sobre política.
y = np.array([int(n["tema"] == "Politica") for n in notas])

prevalencia = y.mean()
piso = max(prevalencia, 1 - prevalencia)
print(f"\nDe cada 100 notas, {prevalencia*100:.0f} son sobre política.")
print(f"EL PISO: un modelo que dice siempre «no es política» acierta "
      f"{piso*100:.1f} % de las veces.")
print(f"        ...y encuentra CERO notas de política, así que su F1 es 0.00.")
print("\nGuardate esos dos números. Todo lo que venga después se compara contra ellos.")

---

## 3. Las tres líneas

Esto es lo que aparece en todos los tutoriales, y funciona:

```python
from transformers import pipeline
clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
clasificador(titulares[:20])
```

Tres líneas y tenés tus titulares clasificados por sentimiento (acá le pasamos los
primeros 20; con `titulares` entero, son los 1471). **La pregunta no es si anda: es si lo
que devolvió mide lo que vos decís que mide.**

La celda de abajo corre eso si hay red, y si no, te muestra exactamente qué habría hecho.

In [ ]:
# ── CAMINO A ── (corrido con red por primera vez en la revisión del 28-09-2026)
if HAY_HF:
    try:
        from transformers import pipeline
        # OJO: el nombre del modelo Y su revisión. Citar un modelo sin versión es como
        # citar un libro sin edición. Volvemos sobre esto en la clase 10.
        MODELO_SENTIMIENTO = "pysentimiento/robertuito-sentiment-analysis"
        clasificador = pipeline("sentiment-analysis", model=MODELO_SENTIMIENTO,
                                revision=REVISION[MODELO_SENTIMIENTO])
        # Detalle honesto: la biblioteca pysentimiento preprocesa los tuits antes de
        # clasificarlos (usuarios, enlaces, emojis). Acá usamos el modelo "pelado", sin
        # ese paso. Con titulares de prensa casi no cambia; con tuits, usá
        # pysentimiento.create_analyzer, que lo hace por vos.
        salida = clasificador(titulares[:20])
        for t, s in zip(titulares[:20], salida):
            print(f"[{s['label']:>3} {s['score']:.2f}]  {t[:78]}")
    except Exception as e:
        print(f"[!] El camino A falló en esta celda ({type(e).__name__}: {str(e)[:100]}).")
        print("    Seguimos igual: la sección 4 vuelve a probar y, si no puede, cae al camino B.")
else:
    print("Sin red no podemos bajar el modelo de sentimiento. Lo que habría corrido es:")
    print("""
    from transformers import pipeline
    clasificador = pipeline("sentiment-analysis",
                            model="pysentimiento/robertuito-sentiment-analysis")
    clasificador(titulares[:20])
    """)
    print("Seguimos con el camino B. No perdemos nada de lo importante.")

### `TODO 1` — la evaluación empieza acá, a mano, en el minuto 5

Corré la celda de abajo. Te imprime 20 titulares con un casillero vacío al lado.

**Leelos y marcá cuáles te parece que un clasificador de sentimiento clasificaría mal**,
o —si corriste el camino A— cuáles clasificó mal de verdad.

*Pista.* Ojo con dos cosas: (a) la **ironía** y la sátira, que se leen como positivas y
no lo son; (b) los titulares que **informan sobre algo malo sin valorarlo**
("murieron 12 personas") — el hecho es negativo, el titular es neutro, y el modelo casi
siempre dice NEG. ¿Eso es un error del modelo, o es que "sentimiento del titular" y
"valencia del hecho" son dos constructos distintos que vos estabas mezclando?

Esa última pregunta es la clase entera.

In [ ]:
print("nº | ¿mal? |  titular")
print("-" * 92)
for i, t in enumerate(titulares[:20]):
    print(f"{i:2d} |  [ ]  |  {t[:76]}")
print("\nContá cuántos marcaste. Si marcaste 3 de 20, tu tasa de error a ojo es 15 %.")
print("Ahora la pregunta incómoda: ¿20 casos alcanzan para afirmar eso?")
print("(No: con 20 casos, 3 errores puede ser 5 % o 35 %. Lo retomamos en el TODO 5")
print(" de la sección 8 y en la receta del final: para evaluar, 200 a 300 casos.)")

---

## 4. Texto → vector

Todo lo que sigue necesita una sola cosa: una función `vectorizar(textos)` que devuelva
una matriz de números, una fila por texto. Eso es un
<span title="embedding de oración">*sentence embedding*</span>: **el texto convertido en
un punto del espacio**, para que "parecerse" sea una distancia y no una corazonada.

Es exactamente el mapa de palabras de la clase 6, con un paso más: el vector de una
oración es el promedio de los vectores de sus palabras.

Una aclaración de método para el camino B: los vectores de palabras se construyen con
**todo** el corpus, test incluido. Es no supervisado —no mira ni una etiqueta— y para
vectores de palabras es práctica estándar. Lo que no se puede hacer nunca es usar el
test para algo que mire la etiqueta: eso sí sería fuga (clase 4).

Escribimos las **dos** implementaciones y elegimos según el camino. Fijate que de acá en
adelante el notebook no vuelve a preguntar en qué camino está: la abstracción tapa la
diferencia, que es justo lo que una buena abstracción tiene que hacer.

In [ ]:
if HAY_HF:
    # ── CAMINO A ── modelo de embeddings de oración multilingüe (anda en castellano).
    MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    try:
        from transformers import AutoTokenizer, AutoModel
        import torch
        tok_hf = AutoTokenizer.from_pretrained(MODELO_EMB, revision=REVISION[MODELO_EMB])
        mod_hf = AutoModel.from_pretrained(MODELO_EMB, revision=REVISION[MODELO_EMB])
        mod_hf.eval()
    except Exception as e:
        # intentar -> si falla, avisar y usar el plan B (el patrón de la sección 1)
        print(f"[!] No pude bajar o cargar {MODELO_EMB} ({type(e).__name__}).")
        print("    Caigo al camino B: todo lo que sigue corre igual.")
        HAY_HF = False
        MOTIVO_SIN_HF = "falló la descarga del modelo de embeddings"

if HAY_HF:
    def vectorizar(textos, lote=64):
        """Cada texto -> un vector, promediando las fichas (mean pooling)."""
        salida = []
        for i in range(0, len(textos), lote):
            trozo = list(textos[i:i + lote])
            entrada = tok_hf(trozo, padding=True, truncation=True,
                             max_length=64, return_tensors="pt")
            with torch.no_grad():
                h = mod_hf(**entrada).last_hidden_state
            mascara = entrada["attention_mask"].unsqueeze(-1).float()
            v = (h * mascara).sum(1) / mascara.sum(1).clamp(min=1e-9)
            salida.append(v.numpy())
        M = np.vstack(salida)
        return M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-9)

    COMO_VECTORIZAMOS = (f"modelo pre-entrenado {MODELO_EMB} "
                         f"(revisión {REVISION[MODELO_EMB][:7]})")

else:
    # ── CAMINO B ── los embeddings del curso: coocurrencia -> PPMI -> SVD.
    # Tres ideas, numpy y nada más. Es lo mismo que hicimos en la clase 6.
    # Se construyen con todos los textos (test incluido): no mira etiquetas.
    corpus_entero = " \n ".join(n["titular"] + " " + n["texto"] for n in notas)

    with utils.medir_tiempo("construir los embeddings con numpy"):
        V, palabras = datos.embeddings_ppmi_svd(
            corpus_entero, dim=100, top=2000, ventana=5, minimo=5)

    _indice = {p: i for i, p in enumerate(palabras)}
    _tok = datos.TokenizadorPalabras(minusculas=True)

    def fichas_conocidas(texto):
        return [w for w in _tok.partir(texto.lower()) if w in _indice]

    def vectorizar(textos):
        """Cada texto -> el promedio de los vectores de sus palabras conocidas."""
        M = np.zeros((len(textos), V.shape[1]), dtype=np.float32)
        for k, t in enumerate(textos):
            ws = fichas_conocidas(t)
            if ws:
                M[k] = V[[_indice[w] for w in ws]].mean(axis=0)
        return M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-9)

    COMO_VECTORIZAMOS = f"PPMI+SVD propios sobre este corpus ({V.shape[1]} dimensiones)"

X = vectorizar(titulares)
print(f"\nVectorizamos con: {COMO_VECTORIZAMOS}")
print(f"Matriz de vectores: {X.shape}   "
      f"({X.shape[0]} titulares, cada uno un punto en {X.shape[1]} dimensiones)")
sin_vector = int((np.linalg.norm(X, axis=1) < 1e-6).sum())
print(f"Titulares que quedaron en el vector cero: {sin_vector} "
      f"(ninguna de sus palabras está en el vocabulario)")
print("\nLos demás están normalizados a norma 1, así que el producto punto YA es la")
print("similitud coseno. Toda la búsqueda de la próxima celda es UNA multiplicación.")

### Buscar por significado

Con los vectores en la mano, buscar es una multiplicación de matrices: comparo el vector
de mi consulta contra todos y me quedo con los más parecidos.

Lo comparamos con la búsqueda de toda la vida —¿contiene la palabra?— que es la que
usaste siempre para armar corpus con listas de términos.

In [ ]:
def buscar_por_significado(consulta, k=6):
    """Los k titulares más parecidos a la consulta. Devuelve [(puntaje, i)]."""
    q = vectorizar([consulta])[0]
    if np.linalg.norm(q) < 1e-6:
        print(f"   (ninguna palabra de «{consulta}» está en el vocabulario: no hay con qué buscar)")
        return []
    similitud = X @ q                      # una sola multiplicación: 1471 comparaciones
    orden = np.argsort(-similitud)[:k]
    return [(float(similitud[i]), int(i)) for i in orden]


def buscar_por_palabra(consulta, k=6):
    """Los que contienen la palabra, sin más. La búsqueda de toda la vida."""
    q = consulta.lower()
    hits = [i for i, t in enumerate(titulares) if q in t.lower()]
    return [(1.0, i) for i in hits[:k]], len(hits)


for consulta in ["fútbol", "vacuna", "protesta"]:
    print("=" * 92)
    print(f"CONSULTA: «{consulta}»")
    porpal, total = buscar_por_palabra(consulta)
    print(f"\n  POR PALABRA CLAVE ({total} en todo el corpus):")
    if not porpal:
        print("     (ninguno)")
    for _, i in porpal:
        print(f"     · {titulares[i][:80]}")
    print("\n  POR SIGNIFICADO:")
    for s, i in buscar_por_significado(consulta):
        marca = "  " if consulta.lower() in titulares[i].lower() else "★ "
        print(f"   {marca}{s:.2f}  {titulares[i][:78]}")
print("=" * 92)
print("★ = el titular NO tiene la palabra que buscaste.")
print()
if not HAY_HF:
    # Estos ejemplos son los del camino B (medidos): con otro modelo salen otros.
    print("Mirá «fútbol»: por palabra clave aparece UNO (un cura con un televisor), y por")
    print("significado aparecen Chivas, la FIFA, el Cruz Azul y Messi, ninguno de los cuales")
    print("dice «fútbol». Eso es lo que tu lista de palabras clave venía perdiendo.")
    print()
    print("Y ahora mirá «protesta», que en este corpus aparece seis veces: entre los ★ hay")
    print("una discoteca que cierra y una nota sobre kebabs. La misma técnica, la misma cara")
    print("de confianza, y basura.")
else:
    print("Leé los ★ uno por uno: son lo que tu lista de palabras clave no habría encontrado.")
    print("¿Cuántos vienen al caso de verdad? Con un modelo pre-entrenado suelen ser más que")
    print("sin red, pero igual aparecen titulares que se PARECEN a la consulta sin ser del tema.")
print("UN RANKING POR PARECIDO NO ES UN CLASIFICADOR VALIDADO: es un buen primer filtro,")
print("y hay que leerlo.")

### `TODO 2` — probalo con algo tuyo

**Consigna.** Elegí **una** categoría de tu propia investigación (conflicto laboral,
género, migración, lo que sea) y buscala de las dos maneras.

Después escribí dos listas:
- **Lo que la búsqueda semántica encontró y la de palabras perdía.** (Ganancia.)
- **Lo que trajo de más y no viene al caso.** (Costo.)

Las dos listas, siempre. Una visualización que sólo muestra los aciertos es publicidad,
no evidencia.

*Pista.* Si te avisa que la palabra no está en el vocabulario (pasa en el camino B, que
sólo conoce las 2000 palabras más frecuentes de este corpus), probá con una más común.
Y acordate de qué es este corpus: prensa en castellano, sobre todo mexicana y española,
de 2018 y de 2020-21. **Un modelo sólo conoce lo que vio** — eso también es un hallazgo.

In [ ]:
# TODO 2: cambiá esta consulta por una de tu tema.
MI_CONSULTA = "protesta"

porpal, total = buscar_por_palabra(MI_CONSULTA, k=8)
sem = buscar_por_significado(MI_CONSULTA, k=8)

print(f"«{MI_CONSULTA}»: {total} por palabra clave, {len(sem)} por significado\n")
nuevos = [i for _, i in sem if MI_CONSULTA.lower() not in titulares[i].lower()]
print(f"La búsqueda semántica trajo {len(nuevos)} que la de palabras NO tenía:")
for i in nuevos:
    print(f"   · {titulares[i][:82]}")

# TODO: escribí acá abajo, en dos listas, cuáles de esos son hallazgos y cuáles basura.
GANANCIA = []   # los índices que SÍ vienen al caso
BASURA = []     # los que no

if GANANCIA or BASURA:
    print(f"\nDe {len(nuevos)} nuevos marcaste {len(GANANCIA)} útiles y {len(BASURA)} basura.")
    if GANANCIA:
        print(f"Precisión de los hallazgos nuevos: "
              f"{len(GANANCIA)/(len(GANANCIA)+len(BASURA)):.0%}")
else:
    print("\n(Todavía no marcaste nada. Leé los de arriba y completá GANANCIA y BASURA.)")

**Tu respuesta:** _(escribí acá qué encontró de más y qué de menos)_

---

## 5. Estrategia A: embeddings + regresión logística

La primera estrategia, y **la que hay que probar siempre primero**: convertís el texto
en vectores (ya está hecho) y le ponés arriba un clasificador de estadística clásica.

Sí: una regresión logística. La misma que ya sabés interpretar. Corre en tu notebook, en
CPU, en un segundo.

Dos cosas de la clase 4 que no se negocian:
- **Partición estratificada**: que train y test tengan la misma proporción de positivos.
- **El test no se toca** hasta el final. Ni para "ver cómo va".

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Partición estratificada. `stratify=y` es lo que mantiene la proporción de positivos.
i_train, i_test = train_test_split(
    np.arange(len(y)), test_size=0.25, stratify=y, random_state=1983)

print(f"train: {len(i_train)} casos, {y[i_train].mean():.1%} positivos")
print(f"test : {len(i_test)} casos, {y[i_test].mean():.1%} positivos   <- la misma proporción")

modelo_a = LogisticRegression(max_iter=2000, class_weight="balanced")
modelo_a.fit(X[i_train], y[i_train])

pred_a = modelo_a.predict(X[i_test])
proba_a = modelo_a.predict_proba(X[i_test])[:, 1]
print(f"\nEntrenado con {len(i_train)} ejemplos. Listo.")

### La función `reportar()`, que te llevás

**Regla de la casa: prohibido reportar una métrica sin su frase en castellano llano.**
Si no podés decir *"de cada 100 notas que el modelo llamó política, 78 lo eran"*, no
entendiste tu propio resultado y no lo podés defender.

Esta función hace las dos cosas de una: la matriz y las frases. Copiala a tus proyectos.

In [ ]:
def reportar(y_true, y_pred, fenomeno="notas sobre política",
             corto="política", dibujar=True):
    """Métricas + matriz + LAS FRASES. Todo junto, porque van juntas."""
    m = g.metricas_binarias(y_true, y_pred)
    prev = float(np.mean(y_true))
    piso_local = max(prev, 1 - prev)

    print(f"--- {fenomeno} ---")
    g.frase_de_metricas(y_true, y_pred, fenomeno=fenomeno)
    print(f"\nEL PISO: decir siempre la clase mayoritaria da accuracy {piso_local:.2f}.")
    print(f"Tu modelo da {m['accuracy']:.2f}. "
          f"Ganancia real sobre el piso: {m['accuracy'] - piso_local:+.2f}.")
    if m["accuracy"] - piso_local < 0.02:
        print("  >> Ojo: eso es prácticamente nada. Reportarlo como éxito sería mentir.")

    if dibujar:
        ax = g.matriz_de_confusion(
            y_true, y_pred,
            etiquetas=[f"no es {corto}", f"es {corto}"],
            nombre_positivo=fenomeno,
            titulo=f"Matriz de confusión — {fenomeno}")
        g.guardar(ax.figure, "figuras/matriz-de-confusion.png")
    return m


metricas_a = reportar(y[i_test], pred_a)

### `TODO 3` — ¿cuánto ganó realmente?

Mirá los dos números que imprimió: la accuracy del modelo y la del piso.

**Escribí la resta.** Y después contestá: si tuvieras que justificarle a un revisor las
horas de trabajo que costó esto, ¿qué número le mostrarías, la accuracy o la ganancia
sobre el piso? ¿Por qué el paper que estás pensando muestra el primero?

---

## 6. Estrategia "cero etiquetas": el diccionario de palabras clave

Antes de las estrategias caras, el piso serio: una lista de palabras escrita a mano.
**Cero etiquetas, cinco minutos de trabajo.**

Es lo que hace muchísima investigación publicada (y es lo que hace LIWC, y los
diccionarios de sentimiento). Con modelos pre-entrenados, este lugar lo ocupa el
*zero-shot*: le describís la categoría con palabras y clasifica sin ejemplos (en este
notebook no lo corremos; en la tabla de decisión va en la misma fila). Distinta
tecnología, misma posición: **cero etiquetas para entrenar, y las mismas 200-300 para
evaluarlo que cualquier otra cosa.**

Y ojo con la trampa que ya conocés desde la clase 1: **la lista la escribimos nosotros**.
Sacá una palabra y la serie cambia.

In [ ]:
DICCIONARIO = (r"gobierno|president|diputad|senad|partido|elecci|"
               r"ministr|congreso|pol[ií]tic|candidat|campa[ñn]a electoral")

pred_dicc = np.array([int(bool(re.search(DICCIONARIO, titulares[i], re.I)))
                      for i in i_test])

metricas_dicc = reportar(y[i_test], pred_dicc, dibujar=False)

print("\n" + "-" * 70)
print("Probá vos: sacale «gobierno» al diccionario y volvé a correr.")
print("El número se mueve. La lista de palabras ES el instrumento de medición,")
print("y lo escribiste vos un martes. (Clase 1, y otra vez la clase 10.)")

---

## 7. La otra punta: aprender el vocabulario desde cero

La estrategia A usa vectores **prestados**: alguien (un modelo pre-entrenado, o los
embeddings del curso) ya había aprendido qué palabras se parecen. Ahora vamos a la otra
punta: **nada prestado**. Una columna por palabra (o par de palabras) de tus titulares
de entrenamiento —eso es **TF-IDF**— y que la regresión aprenda el peso de cada una,
sólo con tus ejemplos.

**Ojo: esto NO es fine-tuning.** El fine-tuning parte de un modelo que ya sabe
castellano y lo sigue entrenando con tus ejemplos: es transferencia. Esto no transfiere
nada. Lo medimos porque es la mejor manera de ver **cuánto vale lo prestado**: muchas
más perillas y ningún conocimiento previo, así que necesita muchos más datos. Clase 4.
(Y además es barato: corre en un segundo, en CPU.)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
A_train = tfidf.fit_transform([titulares[i] for i in i_train])
A_test = tfidf.transform([titulares[i] for i in i_test])

print(f"TF-IDF: {A_train.shape[1]} columnas (una por palabra o par de palabras).")
print(f"Estrategia A tenía {X.shape[1]}. O sea: {A_train.shape[1] // X.shape[1]}x "
      f"más perillas que ajustar con los mismos {len(i_train)} ejemplos.")

modelo_c = LogisticRegression(max_iter=2000, class_weight="balanced")
modelo_c.fit(A_train, y[i_train])
pred_c = modelo_c.predict(A_test)

metricas_c = reportar(y[i_test], pred_c, dibujar=False)

### La tabla comparativa honesta

Las cuatro opciones, con el mismo test, en la misma tabla. **Esta tabla es el entregable
de la clase**: es lo que vas a poner en tu paper.

In [ ]:
prev_test = float(y[i_test].mean())
filas = [
    ("el piso (decir siempre «no»)", max(prev_test, 1 - prev_test), 0.0, 0.0, 0.0, "0 etiquetas"),
    ("diccionario de palabras clave", metricas_dicc["accuracy"], metricas_dicc["precision"],
     metricas_dicc["recall"], metricas_dicc["f1"], "0 etiquetas"),
    ("A · embeddings + logística", metricas_a["accuracy"], metricas_a["precision"],
     metricas_a["recall"], metricas_a["f1"], f"{len(i_train)} etiquetas"),
    ("desde cero · TF-IDF + logística", metricas_c["accuracy"], metricas_c["precision"],
     metricas_c["recall"], metricas_c["f1"], f"{len(i_train)} etiquetas"),
]

print(f"{'estrategia':<32}{'acc':>7}{'prec':>7}{'rec':>7}{'F1':>7}   costo")
print("-" * 84)
for nombre, acc, prec, rec, f1, costo in filas:
    print(f"{nombre:<32}{acc:>7.2f}{prec:>7.2f}{rec:>7.2f}{f1:>7.2f}   {costo}")
print("-" * 84)
mejor = max(filas[1:], key=lambda f: f[4])
print(f"Gana por F1: {mejor[0]}  ({mejor[4]:.2f})")
print("\nFalta una fila, a propósito: el fine-tuning. No está medido en esta tabla")
print("(la celda opcional de abajo lo corre en el camino A). La fila «desde cero» NO lo")
print("reemplaza: es la otra punta, la que no aprovecha nada prestado.")

### El fine-tuning de verdad (opcional), y el `Trainer` que lo esconde

El fine-tuning agarra un modelo pre-entrenado y **lo sigue entrenando entero** con tus
ejemplos, con una capa nueva arriba para tu distinción. Con `Trainer` de Hugging Face se
escribe así (BETO, con su revisión fijada):

```python
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

modelo = AutoModelForSequenceClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-uncased",                       # BETO
    revision="d1c9c4565c9d6731e57ed7f027b802697bad861e", num_labels=2)

args = TrainingArguments(output_dir="salida", num_train_epochs=3,
                         per_device_train_batch_size=16, learning_rate=2e-5,
                         eval_strategy="epoch", seed=1983)
Trainer(model=modelo, args=args,
        train_dataset=ds_train, eval_dataset=ds_val).train()
```

**Adentro de ese `.train()` no pasa nada que no hayas escrito vos en la clase 4.** La
celda de abajo lo hace a mano, sin `Trainer`, con las cinco estaciones a la vista. Está
**apagada**: si estás en el camino A, poné `CORRER_FINE_TUNING = True` (con 200
etiquetas tarda entre 10 y 15 segundos en CPU, sobre MiniLM multilingüe).

**Lo que medimos en la revisión** (MiniLM multilingüe, 3 épocas, tres semillas por
tamaño, el mismo test de 368): F1 **0,53** con 25 etiquetas, **0,59** con 200 y **0,67**
con 800. O sea: con pocas etiquetas le gana por lejos a aprender desde cero (0,14 con
25), anda parecido a embeddings + logística y un poco mejor con 400-800, y **es
inestable**: con 50 etiquetas, según la semilla, dio de 0,22 a 0,57. Con modelos más
grandes y más datos suele ganar más; hay que medirlo en tu corpus, no suponerlo.

In [ ]:
# ── CAMINO A, OPCIONAL ── fine-tuning a mano: las cinco estaciones de la clase 4.
CORRER_FINE_TUNING = False   # True para correrlo (sólo en el camino A)
N_FT = 200                   # cuántas etiquetas usa (probá 50 y 800)

if CORRER_FINE_TUNING and not HAY_HF:
    print("El fine-tuning necesita bajar el modelo pre-entrenado: sólo corre en el camino A.")
elif CORRER_FINE_TUNING:
    import torch
    from transformers import AutoModelForSequenceClassification

    torch.manual_seed(1983)
    rng_ft = np.random.default_rng(1983)
    prev_tr = y[i_train].mean()
    n_pos_ft = max(2, int(round(N_FT * prev_tr)))
    muestra_ft = np.concatenate([
        rng_ft.choice([i for i in i_train if y[i] == 1], n_pos_ft, replace=False),
        rng_ft.choice([i for i in i_train if y[i] == 0], N_FT - n_pos_ft, replace=False)])

    # El modelo pre-entrenado + una capa nueva arriba (2 salidas: sí / no). Por eso
    # transformers avisa que `classifier` está MISSING: es la capa nueva, y es esperable.
    modelo_ft = AutoModelForSequenceClassification.from_pretrained(
        MODELO_EMB, revision=REVISION[MODELO_EMB], num_labels=2)
    optimizador = torch.optim.AdamW(modelo_ft.parameters(), lr=5e-5)
    peso = torch.tensor([1.0, float((1 - prev_tr) / prev_tr)])   # como class_weight="balanced"

    with utils.medir_tiempo(f"fine-tuning con {N_FT} etiquetas"):
        for epoca in range(3):
            modelo_ft.train()
            orden = rng_ft.permutation(muestra_ft)
            for k in range(0, len(orden), 16):
                lote = orden[k:k + 16]
                entrada = tok_hf([titulares[i] for i in lote], padding=True,
                                 truncation=True, max_length=64, return_tensors="pt")
                prediccion = modelo_ft(**entrada)                            # 1. PREDECIR
                perdida = torch.nn.functional.cross_entropy(                 # 2. COMPARAR
                    prediccion.logits, torch.tensor(y[lote]), weight=peso)
                optimizador.zero_grad()                                      # 3. BORRAR
                perdida.backward()                                           # 4. ANOTAR
                optimizador.step()                                           # 5. MOVER
            print(f"   época {epoca + 1}: loss del último lote {perdida.item():.3f}")

    modelo_ft.eval()
    pred_ft = []
    with torch.no_grad():
        for k in range(0, len(i_test), 64):
            entrada = tok_hf([titulares[i] for i in i_test[k:k + 64]], padding=True,
                             truncation=True, max_length=64, return_tensors="pt")
            pred_ft.append(modelo_ft(**entrada).logits.argmax(-1).numpy())
    m_ft = g.metricas_binarias(y[i_test], np.concatenate(pred_ft))
    print(f"\nFine-tuning con {N_FT} etiquetas: F1 en test {m_ft['f1']:.2f} "
          f"(precisión {m_ft['precision']:.2f}, recall {m_ft['recall']:.2f}).")
    print("Comparalo con la curva de la sección 8 en el mismo tamaño. Y cambiá la semilla:")
    print("con pocas etiquetas el fine-tuning se mueve mucho de una corrida a otra.")
else:
    print("Celda opcional apagada (CORRER_FINE_TUNING = False).")
    print("Los números que medimos están en el texto de arriba.")

---

## 8. La curva: ¿cuántas etiquetas hacen falta?

Esta es la decisión práctica que vas a tomar de verdad, y no depende de qué técnica está
de moda: depende de **cuántas horas de codificación podés pagar**.

Entrenamos las dos estrategias con 25, 50, 100, 200, 400 y 800 ejemplos, cinco veces cada
una con semillas distintas, y miramos qué pasa.

Ojo con qué compara esta curva: **vectores prestados contra nada prestado**. Dice cuánto
vale lo que ya venía aprendido cuando tenés pocas etiquetas. **No** dice nada sobre el
fine-tuning, que también parte de algo prestado (eso está en la celda opcional de la
sección 7).

In [ ]:
from sklearn.metrics import f1_score

TAMANIOS = [25, 50, 100, 200, 400, 800]
REPETICIONES = 5

pos = np.array([i for i in i_train if y[i] == 1])
neg = np.array([i for i in i_train if y[i] == 0])
prev_train = y[i_train].mean()

curva = {"embeddings": [], "tfidf": []}
with utils.medir_tiempo("la curva entera"):
    for n in TAMANIOS:
        f_emb, f_tfi = [], []
        for s in range(REPETICIONES):
            rng = np.random.default_rng(1983 + s)
            n_pos = max(2, int(round(n * prev_train)))
            muestra = np.concatenate([
                rng.choice(pos, min(n_pos, len(pos)), replace=False),
                rng.choice(neg, n - n_pos, replace=False)])

            m1 = LogisticRegression(max_iter=2000, class_weight="balanced")
            m1.fit(X[muestra], y[muestra])
            f_emb.append(f1_score(y[i_test], m1.predict(X[i_test]), zero_division=0))

            tf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
            Am = tf.fit_transform([titulares[i] for i in muestra])
            m2 = LogisticRegression(max_iter=2000, class_weight="balanced")
            m2.fit(Am, y[muestra])
            f_tfi.append(f1_score(y[i_test],
                                  m2.predict(tf.transform([titulares[i] for i in i_test])),
                                  zero_division=0))
        curva["embeddings"].append((np.mean(f_emb), np.std(f_emb)))
        curva["tfidf"].append((np.mean(f_tfi), np.std(f_tfi)))
        print(f"  n={n:4d}   embeddings {np.mean(f_emb):.3f}   "
              f"vocabulario desde cero {np.mean(f_tfi):.3f}")

In [ ]:
fig, ax = g.figura(9, 5,
                   titulo="¿Cuántas etiquetas hacen falta? (corpus de prensa del curso)",
                   xlabel="casos etiquetados a mano (escala log)", ylabel="F1")

# Cada curva con su color Y su forma (redondos contra cuadrados): que no dependa del color.
for clave, color, estilo, etiqueta in [
        ("embeddings", g.MODELO, "o-", "embeddings + regresión logística (prestado)"),
        ("tfidf", g.PARAMETRO, "s-.", "aprender el vocabulario desde cero (nada prestado)")]:
    medias = np.array([m for m, _ in curva[clave]])
    desvios = np.array([s for _, s in curva[clave]])
    ax.plot(TAMANIOS, medias, estilo, color=color, lw=2.5, label=etiqueta)
    ax.fill_between(TAMANIOS, medias - desvios, medias + desvios, color=color, alpha=.15)

ax.axhline(metricas_dicc["f1"], color=g.DATOS, ls="--", lw=2,
           label=f"cero etiquetas: diccionario ({metricas_dicc['f1']:.2f})")
ax.set_xscale("log")
ax.set_xticks(TAMANIOS)
ax.set_xticklabels([str(t) for t in TAMANIOS])
ax.minorticks_off()          # sin esto el eje log llena de 3x10^1, 4x10^1...
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=9)
g.guardar(fig, "figuras/curva-de-etiquetas.png")
print("Figura guardada en figuras/curva-de-etiquetas.png")

### `TODO 4` y `TODO 5` — de la curva a una decisión de presupuesto

Mirá el gráfico y contestá tres cosas por escrito:

1. **¿Dónde deja de mejorar mucho?** Encontrá el punto a partir del cual duplicar las
   etiquetas casi no mueve el F1.
2. **Traducilo a horas.** Estimá 30 segundos por titular. ¿Cuántas horas de codificación
   son las etiquetas de ese punto? ¿Y las del doble?
3. **La pregunta profesional, no la técnica.** Escribí la recomendación de dos oraciones
   que le darías a un colega con tu mismo corpus y tu mismo presupuesto. No "usá
   BERT": algo que incluya cuántos casos etiquetar y con qué estrategia.

*Pista para la 2.* 200 casos × 30 s ≈ 1,7 horas. Y sumale un 20 % doblemente codificado
para poder reportar el acuerdo entre codificadores (sección 12): otros 20 minutos.

In [ ]:
# TODO 5: completá y corré. El assert te avisa amablemente si algo no cierra.
MI_PRESUPUESTO_EN_ETIQUETAS = 200      # ¿cuántos casos podés codificar de verdad?
SEGUNDOS_POR_CASO = 30

horas = MI_PRESUPUESTO_EN_ETIQUETAS * SEGUNDOS_POR_CASO / 3600
horas_doble = MI_PRESUPUESTO_EN_ETIQUETAS * 0.2 * SEGUNDOS_POR_CASO / 3600

try:
    assert MI_PRESUPUESTO_EN_ETIQUETAS > 0, (
        "Poné un número mayor que cero. Aunque sea 50: con cero etiquetas no podés "
        "ni medir si tu clasificador anda, y sin eso no hay investigación.")
    assert MI_PRESUPUESTO_EN_ETIQUETAS >= 100, (
        f"Con {MI_PRESUPUESTO_EN_ETIQUETAS} casos vas a poder entrenar, pero el test "
        "te queda tan chico que tus métricas van a tener un intervalo de confianza "
        "enorme. No está mal como punto de partida: anotalo como limitación y seguí.")
    print(f"Etiquetar {MI_PRESUPUESTO_EN_ETIQUETAS} casos = {horas:.1f} horas de "
          f"codificación (+{horas_doble*60:.0f} min para el 20 % doblemente codificado).")
except AssertionError as e:
    print("Pará un segundo:\n  " + str(e))

i_cercano = int(np.argmin([abs(t - MI_PRESUPUESTO_EN_ETIQUETAS) for t in TAMANIOS]))
print(f"\nCon ~{TAMANIOS[i_cercano]} etiquetas, en este corpus:")
print(f"   embeddings + logística   F1 {curva['embeddings'][i_cercano][0]:.2f}")
print(f"   vocabulario desde cero   F1 {curva['tfidf'][i_cercano][0]:.2f}")
print(f"   diccionario (0 etiquetas) F1 {metricas_dicc['f1']:.2f}")

---

## 9. El umbral: elegir qué error preferís

Hasta acá usamos `predict()`, que corta en 0,5 porque sí. Pero el modelo no devuelve un
"sí" o un "no": devuelve un **puntaje**. Vos elegís dónde cortar.

Y esa elección no es técnica: **es una decisión de investigación**.
- ¿Vas a *leer* todo lo que el modelo marque? Te importa la **precisión** (que no te haga
  perder tiempo con basura).
- ¿Vas a estimar *cuántos* casos hay? Te importan las dos, balanceadas.
- ¿No te podés permitir perder ninguno? Te importa el **recall**, y vas a pagar con
  falsos positivos.

Antes de correr: **apostá**. Si bajás el umbral, ¿pueden subir la precisión y el recall
las dos a la vez?

Y esta vez lo hacemos bien: **el umbral se elige en validación, no en test.** Como no
apartamos una partición de validación, usamos validación cruzada sobre train: cada caso
de train recibe el puntaje de un modelo que no lo vio. El test se mira una sola vez, al
final, con el umbral ya elegido.

> Si todavía no la abriste, éste es el momento de `viz/matriz-de-confusion.html`.
> El gráfico de abajo es la versión estática de lo que ahí podés arrastrar con el dedo.

In [ ]:
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Puntajes de VALIDACIÓN: validación cruzada en 5 pliegues, sólo con train.
pliegues = StratifiedKFold(n_splits=5, shuffle=True, random_state=1983)
proba_val = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"),
                              X[i_train], y[i_train], cv=pliegues,
                              method="predict_proba")[:, 1]
y_val = y[i_train]

umbrales = np.linspace(0.05, 0.95, 37)
precisiones, recalls, efes = [], [], []
for u in umbrales:
    p = (proba_val >= u).astype(int)
    precisiones.append(precision_score(y_val, p, zero_division=0))
    recalls.append(recall_score(y_val, p, zero_division=0))
    efes.append(f1_score(y_val, p, zero_division=0))

fig, ax = g.figura(9, 4.5, titulo="Movés el umbral y elegís qué error preferís (validación)",
                   xlabel="umbral de decisión", ylabel="de 0 a 1")
ax.plot(umbrales, precisiones, color=g.MODELO, lw=2.5, label="precisión")
ax.plot(umbrales, recalls, color=g.VAL, lw=2.5, ls="-.", label="recall")
ax.plot(umbrales, efes, color=g.ACENTO, lw=2, ls="--", label="F1")
u_mejor = float(umbrales[int(np.argmax(efes))])
ax.axvline(u_mejor, color=g.PARAMETRO, lw=1.5, ls=":")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
g.guardar(fig, "figuras/umbral.png")

print(f"En validación, el F1 se maximiza con el umbral en {u_mejor:.2f} (F1 = {max(efes):.2f}).")

# ¿Suben las dos en algún tramo? Bajamos el umbral paso a paso y contamos.
suben_las_dos = [(umbrales[k + 1], umbrales[k]) for k in range(len(umbrales) - 1)
                 if precisiones[k] > precisiones[k + 1] and recalls[k] > recalls[k + 1]]
print("\nAl bajar el umbral, el recall nunca baja: sólo sube o se queda igual.")
print("La precisión TIENDE a bajar, pero no siempre: si el caso que entra es un acierto,")
print(f"sube. En este gráfico hay {len(suben_las_dos)} tramo(s) donde, al bajar el umbral un paso,")
print("suben las dos. Son saltitos, no una tendencia: no hay un umbral que mejore las dos")
print("de forma sostenida. Elegir el umbral es elegir qué error preferís.")

# Y recién ahora, UNA vez, el test con el umbral elegido en validación.
m_umbral = g.metricas_binarias(y[i_test], (proba_a >= u_mejor).astype(int))
print(f"\nTEST, una sola vez, con el umbral {u_mejor:.2f}: F1 = {m_umbral['f1']:.2f} "
      f"(precisión {m_umbral['precision']:.2f}, recall {m_umbral['recall']:.2f}).")

# Lo que NO hay que hacer, para que veas cuánto infla:
f1_mirando_test = [f1_score(y[i_test], (proba_a >= u).astype(int), zero_division=0)
                   for u in umbrales]
u_trampa = float(umbrales[int(np.argmax(f1_mirando_test))])
print(f"Si lo hubieras elegido mirando el test, habrías puesto {u_trampa:.2f} y reportado")
print(f"F1 = {max(f1_mirando_test):.2f}. Acá la diferencia es chica; con tests chicos puede no serlo,")
print("y en los dos casos es ajustar contra el test: hacer trampa. (Clase 4.)")

---

## 10. Mirá los errores. En serio, leelos.

Leer 30 errores del modelo te enseña más sobre **tu propia definición operativa** que
leer 300 aciertos.

In [ ]:
falsos_positivos = [i for k, i in enumerate(i_test) if pred_a[k] == 1 and y[i] == 0]
falsos_negativos = [i for k, i in enumerate(i_test) if pred_a[k] == 0 and y[i] == 1]

print(f"{len(falsos_positivos)} falsos positivos y {len(falsos_negativos)} falsos negativos.\n")

print("=" * 92)
print("FALSOS POSITIVOS — el modelo dijo «política» y la etiqueta dice que no")
print("=" * 92)
for i in falsos_positivos[:15]:
    print(f"  [{notas[i]['tema']:<13}] {titulares[i][:72]}")

print("\n" + "=" * 92)
print("FALSOS NEGATIVOS — era política y el modelo no la vio")
print("=" * 92)
for i in falsos_negativos[:15]:
    print(f"  [{notas[i]['fuente'][:13]:<13}] {titulares[i][:72]}")

### `TODO 6` (el importante de la clase)

Leé los 30 errores de arriba —**leelos de verdad, uno por uno**— y escribí tres cosas:

**(a) ¿Tienen algo en común?** ¿Son de un tema particular? ¿De un medio? ¿Son cortos?
¿Están en mayúsculas?

**(b) ¿Alguno está en realidad *bien* clasificado y la etiqueta era la equivocada?**
Vas a encontrar varios. Una nota sobre una decisión del gobierno etiquetada como
"Sociedad", una de Covid que también es política sanitaria. **Cuando eso pasa, tu F1
está midiendo el desacuerdo entre vos y quien etiquetó el corpus**, no la calidad del
modelo.

**(c) ¿Qué te dice esto sobre tu definición de la categoría?**
Si vos y el corpus no se ponen de acuerdo en qué es "política", ningún modelo lo va a
resolver: **el problema es conceptual, no computacional.**

Este ejercicio, hecho una vez, te cambia la relación con las métricas para siempre.

**Tu respuesta:** _(escribí acá (a), (b) y (c))_

---

## 11. La trampa de este corpus (y de casi todos)

Cambiamos de tarea un momento. Este corpus trae otra etiqueta: **verdadera / falsa**.
Suena a que se podría entrenar un detector de desinformación. Vamos a entrenarlo, a
mirar el número lindo, y después a mirar de dónde salió.

In [ ]:
y_falso = np.array([n["es_falso"] for n in notas])
fuentes = [n["fuente"] for n in notas]
textos_largos = [n["titular"] + " . " + n["texto"] for n in notas]


def entrenar_verdad_falso(idx_tr, idx_te):
    tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    A = tf.fit_transform([textos_largos[i] for i in idx_tr])
    m = LogisticRegression(max_iter=2000, C=4).fit(A, y_falso[idx_tr])
    p = m.predict(tf.transform([textos_largos[i] for i in idx_te]))
    return g.metricas_binarias(y_falso[idx_te], p)


# (1) Partición al azar, como se hace en todos los tutoriales.
tr_azar, te_azar = train_test_split(np.arange(len(y_falso)), test_size=0.25,
                                    stratify=y_falso, random_state=1983)
m_azar = entrenar_verdad_falso(tr_azar, te_azar)

# (2) Partición POR MEDIO: ningún diario aparece en train y en test a la vez.
tr_g, va_g, te_g = datos.particion_por_grupo(fuentes, semilla=1983)
m_grupo = entrenar_verdad_falso(np.array(tr_g), np.array(te_g))

def piso_de(idx):
    """La accuracy de decir siempre la clase mayoritaria DE ESE test."""
    p = float(y_falso[idx].mean())
    return max(p, 1 - p)


print(f"{'partición':<28}{'piso':>7}{'acc':>7}{'F1':>7}")
print("-" * 51)
print(f"{'al azar':<28}{piso_de(te_azar):>7.2f}{m_azar['accuracy']:>7.2f}{m_azar['f1']:>7.2f}")
print(f"{'por medio (sin fuga)':<28}{piso_de(te_g):>7.2f}{m_grupo['accuracy']:>7.2f}{m_grupo['f1']:>7.2f}")
print("-" * 51)
print(f"La caída es de {m_azar['f1'] - m_grupo['f1']:.2f} de F1. Y fijate el piso: partiendo por medio,")
print("el test queda con otra proporción de falsas, y el modelo ni siquiera le gana al bobo.")

In [ ]:
# ¿Por qué se cae tanto? Miremos quién publica qué.
por_medio = {}
for n in notas:
    por_medio.setdefault(n["fuente"], Counter())[n["categoria"]] += 1
una_sola = sum(1 for c in por_medio.values() if len(c) == 1)

print(f"{len(por_medio)} medios distintos. {una_sola} de ellos "
      f"({una_sola/len(por_medio):.0%}) aparecen con UNA SOLA clase.\n")

print("Los cinco medios que más notas «falsas» aportan:")
for medio, cuantas in Counter(n["fuente"] for n in notas if n["es_falso"]).most_common(5):
    print(f"   {medio:<20} {cuantas:>4}")
print("\nLos cinco que más «verdaderas»:")
for medio, cuantas in Counter(n["fuente"] for n in notas if not n["es_falso"]).most_common(5):
    print(f"   {medio:<20} {cuantas:>4}")

### Lo que hay que decir en voz alta

**El Dizque y El Ruinaversal son sitios de sátira.** Publican noticias falsas *a
propósito*, y con eso hacen humor. No son sitios de desinformación.

Entonces, ¿qué aprendió el clasificador que sacó un F1 lindo con la partición al azar?
Casi seguro esto: **"¿está escrito con el estilo de un sitio de sátira mexicano?"**.

Y funciona. Y es reproducible. Y no es un detector de desinformación: es un
**detector de registro humorístico**. Ese hueco entre *lo que querés medir* y *lo que el
modelo efectivamente usa* tiene nombre —problema de validez de constructo— y es la clase
que viene entera.

La prueba está en la tabla de arriba: en cuanto le sacás la posibilidad de reconocer al
medio (partiendo por fuente), el modelo se cae por debajo del piso de ese test. **Un modelo que
depende de una variable que no vas a tener en tus datos nuevos no sirve para nada.**

> **Regla para llevarse:** entrená siempre dos veces, con y sin el atajo sospechoso
> disponible, y mostrá la diferencia. **Si no cae, algo está mal en tu evaluación.**

### Y una trampa más: tema y época

Este corpus no trae fecha, pero trae la partición original, y cada partición es **una
recolección distinta**: `train` y `dev` son de enero a julio de 2018; `test`, de
noviembre de 2020 a marzo de 2021. Mirá dónde caen las notas de Covid-19:

In [ ]:
import csv
from ssdl.datos import ruta_de_muestras

with open(ruta_de_muestras("prensa", "titulares-prensa-es.csv"), encoding="utf-8") as f:
    filas_csv = list(csv.DictReader(f))
# Vamos fila por fila, en el mismo orden que `notas` (salen del mismo CSV).
# No usamos el `id` como llave: se repite entre particiones (cada archivo original
# numeraba desde 1). Otro detalle de los que muerden si no mirás.
assert [r["titular"] for r in filas_csv] == titulares
epoca = ["2020-21" if r["particion_original"] == "test" else "2018" for r in filas_csv]

print(f"{'tema':<12}{'2018':>7}{'2020-21':>9}")
for tema in ["Covid-19", "Politica", "Sociedad"]:
    c = Counter(e for n, e in zip(notas, epoca) if n["tema"] == tema)
    print(f"{tema:<12}{c['2018']:>7}{c['2020-21']:>9}")
print(f"{'(todas)':<12}{epoca.count('2018'):>7}{epoca.count('2020-21'):>9}")

Las 192 notas de Covid-19 son **todas** de la segunda recolección. **Tema y época están
confundidos**: un modelo que aprende a reconocer "Covid-19" también está aprendiendo a
reconocer "2020-21", y si lo usás para separar temas en otro período puede estar
separando épocas. Es otro atajo, y es el tipo de confusor que la clase 10 te enseña a
buscar antes de que te lo encuentre un revisor.

---

## 12. El techo humano: el acuerdo entre codificadores

La pregunta que un cientista social hace mejor que nadie en un equipo de ML:
**¿cuánto coinciden dos personas entrenadas codificando lo mismo?**

Porque ese es tu techo realista. Y hay que compararlo **en la misma métrica**: si el F1 de
un codificador contra el otro es 0,71, un modelo que saca F1 0,95 contra las etiquetas de
*uno* de ellos casi seguro no es mejor que un humano: **está midiendo otra cosa, o hay
una fuga.** (No es imposible que un modelo coincida con un codificador más que el otro
codificador —promedia el ruido de cada uno—, pero es raro, y hay que demostrarlo.)

> ⚠️ **Los dos codificadores de este archivo son SIMULADOS** y está escrito en el nombre
> del archivo. No hubo dos personas codificando: se generaron con `random.Random(1983)`.
> Sirven para practicar la cuenta y discutir qué significa el número. **No sirven para
> afirmar nada sobre la confiabilidad del corpus real**, y reportarlos como si fueran
> humanos sería fraude.

In [ ]:
import csv
from ssdl.datos import ruta_de_muestras

ruta = ruta_de_muestras("prensa", "doble-codificacion-SIMULADA-100.csv")
with open(ruta, encoding="utf-8") as f:
    filas = [r for r in csv.DictReader(f) if not r["id"].startswith("#")]

A = np.array([int(r["codificador_A_SIMULADO"]) for r in filas])
B = np.array([int(r["codificador_B_SIMULADO"]) for r in filas])

acuerdo = float((A == B).mean())

# Kappa de Cohen a mano, que son tres líneas y conviene verlas.
p_a = acuerdo                                   # acuerdo observado
p_e = (A.mean() * B.mean()) + ((1 - A.mean()) * (1 - B.mean()))   # acuerdo esperado por azar
kappa = (p_a - p_e) / (1 - p_e)

# El techo en la MISMA métrica que usás para el modelo: F1 de un codificador contra
# el otro (da lo mismo cuál tomás como "verdad": el F1 es simétrico).
f1_humano = g.metricas_binarias(A, B)["f1"]

print(f"{len(filas)} casos codificados por dos personas (SIMULADAS), que marcaron si la")
print("nota es falsa (la misma etiqueta de la sección 11).")
print(f"Acuerdo observado : {acuerdo:.2f}   (coinciden en {acuerdo:.0%} de los casos)")
print(f"Acuerdo por azar  : {p_e:.2f}   (lo que sacarían tirando una moneda cargada)")
print(f"KAPPA DE COHEN    : {kappa:.2f}")
print()
print("Cómo se lee un kappa según Landis y Koch (1977). Es una convención, no una ley:")
print("   < 0 pobre  ·  0-0.20 leve  ·  0.21-0.40 regular  ·  0.41-0.60 moderado")
print("   0.61-0.80 sustancial  ·  0.81-1 casi perfecto")
print()
print(f"F1 DE UN CODIFICADOR CONTRA EL OTRO: {f1_humano:.2f}")
print("Ése es el techo contra el que se compara el F1 de un modelo evaluado contra UNO")
print(f"de ellos. El kappa ({kappa:.2f}) NO: es otra escala (corregida por azar), y compararlo")
print("con un F1 es mezclar peras con manzanas.")
print(f"Si tu modelo pasa claramente de {f1_humano:.2f} contra las etiquetas de uno solo, no")
print("festejes todavía: buscá la fuga o el atajo antes de creerlo.")

### `TODO 7` (criterio, no código)

Situación real, y te va a pasar:

> El kappa entre tus dos codificadores es **0,71**, y el F1 de uno contra el otro, **0,80**.
> Tu modelo, evaluado contra las etiquetas de **uno solo** de ellos, saca **F1 0,88**.

**Nombrá dos explicaciones posibles y decí cómo distinguirías entre ellas.**

*Para arrancar:* una es alegre y la otra no. Pensá qué pasaría si evaluaras el mismo
modelo contra las etiquetas del **otro** codificador, y qué pasaría si evaluaras a un
codificador humano contra el otro con la misma métrica que usás para el modelo.

**Tu respuesta:** _(escribí acá)_

---

## 13. Para llevarte: `clasificar_corpus()`

Esto es lo concreto. Una función lista para que le pases **tus** textos.
Fijate que devuelve el puntaje, no sólo la decisión: sin el puntaje no podés mover el
umbral, y sin mover el umbral no podés elegir qué error preferís.

In [ ]:
def clasificar_corpus(textos, modelo=None, umbral=0.5):
    """
    Clasifica una lista de textos con el modelo entrenado acá.

    Devuelve una lista de dicts: {texto, puntaje, prediccion}.
    El puntaje va de 0 a 1: es la confianza del modelo, NO una probabilidad
    calibrada y NO una medida de verdad. Mové el umbral según qué error preferís.
    """
    modelo = modelo or modelo_a
    M = vectorizar(list(textos))
    puntajes = modelo.predict_proba(M)[:, 1]
    return [{"texto": t, "puntaje": float(p), "prediccion": int(p >= umbral)}
            for t, p in zip(textos, puntajes)]


# (a) Titulares parecidos a los del corpus con el que se entrenó.
print("DENTRO DEL DOMINIO (prensa en castellano de 2018 y 2020-21, sobre todo mexicana")
print("y española: lo que vio):")
for r in clasificar_corpus([
        "El presidente presentó su informe de gobierno ante el Congreso",
        "Cinco heridos en un choque múltiple en la autopista",
        "El PAN y Morena se enfrentan por la reforma electoral"]):
    print(f"  [{'SÍ' if r['prediccion'] else 'no'}  {r['puntaje']:.2f}]  {r['texto']}")

# (b) Titulares argentinos de 2026: otro país, otra coyuntura, otro vocabulario.
print("\nFUERA DEL DOMINIO (prensa argentina: otro país, otra coyuntura, otro vocabulario):")
fuera = ["El oficialismo y la oposición se cruzaron por el presupuesto",
         "Paro de colectivos en el AMBA por reclamo salarial",
         "Récord de temperatura en el sur: 41 grados en pleno enero"]
for r in clasificar_corpus(fuera):
    print(f"  [{'SÍ' if r['prediccion'] else 'no'}  {r['puntaje']:.2f}]  {r['texto']}")

if not HAY_HF:
    print("\n¿Por qué se equivoca abajo? Mirá qué palabras cayeron fuera del vocabulario:")
    for t in fuera:
        conocidas = set(fichas_conocidas(t))
        todas = [w for w in _tok.partir(t.lower()) if w.isalpha()]
        perdidas = [w for w in todas if w not in conocidas]
        print(f"   «{t[:44]}…»  no conoce: {', '.join(perdidas) or '(ninguna)'}")
    print("\nESTO NO ES UN BUG: es el límite del modelo, y es reportable.")
    print("Un clasificador entrenado con prensa mexicana y española de 2018 y 2020-21")
    print("no sabe qué es el AMBA ni qué significa «oficialismo» en Argentina. Si lo")
    print("aplicás a otro país u otra época SIN revalidar, vas a publicar una serie")
    print("temporal que mide vocabulario.")
    print("(Y eso, exactamente, es el gancho de la clase 10.)")
else:
    print("\nCon un modelo pre-entrenado quizás acierte más afuera del dominio: vio mucho más")
    print("castellano. Pero «quizás» no se publica: fuera del dominio, se revalida con")
    print("etiquetas propias. (Clase 10.)")

print("\nY la ficha del modelo, que va SÍ O SÍ en la sección de métodos:")
print(f"   vectorización : {COMO_VECTORIZAMOS}")
print(f"   clasificador  : regresión logística (class_weight='balanced'), scikit-learn")
print(f"   entrenamiento : {len(i_train)} casos, partición estratificada, semilla 1983")
print(f"   evaluación    : {len(i_test)} casos held-out, nunca usados para elegir nada")
print(f"                   (el umbral de la sección 9 se eligió en validación cruzada)")
print(f"   F1 en test    : {metricas_a['f1']:.2f} con umbral 0.5  (piso: 0.00)")

---

## 14. Qué te llevás

**Las tres cosas de hoy:**

1. **Hacerlo andar es fácil; saber si anda es tu trabajo.** Tres líneas bajan un modelo.
   La muestra etiquetada a mano, el piso, la matriz de confusión y la lectura de los
   errores son lo que convierte eso en investigación.
2. **La estrategia se elige por cuántas etiquetas podés producir, no por cuál técnica
   está de moda.** Con pocas, lo simple gana; y a veces el diccionario de cinco minutos
   le gana a todo, y hay que decirlo.
3. **Toda métrica va con su frase en castellano y su piso al lado.** Si no podés decir
   *"de cada 100 que el modelo llamó X, N lo eran"*, no lo entendiste. Y si tu clase
   mayoritaria es el 92 %, un 93 % de accuracy es un desastre que suena bien.

**El vocabulario nuevo** (también en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *transfer learning* | aprovechar lo que un modelo ya aprendió en otra tarea |
| *fine-tuning* | seguir entrenando ese modelo con tus propios ejemplos |
| *freezing* / congelar capas | dejar quietas las capas de abajo y mover sólo las de arriba |
| *zero-shot* | clasificar describiendo la categoría con palabras, sin ejemplos para entrenar (para evaluar, sí) |
| *pipeline* | el atajo de Hugging Face: texto entra, etiqueta sale |
| *sentence embedding* | la oración entera convertida en un punto del espacio |
| *precision* | de lo que el modelo llamó X, cuánto era X |
| *recall* | de todo lo que era X, cuánto encontró |
| *baseline* | el piso: cuánto saca no hacer nada |
| *kappa* | cuánto coinciden dos codificadores, descontando el azar |
| *threshold* / umbral | dónde cortás el puntaje para decir «sí» |

**En tu propia investigación, desde ya.** La receta de siete pasos. No es tarea para la
clase que viene: es un proyecto de mediano plazo (sólo etiquetar 300 casos son unas 2,5
horas, más la otra persona que codifica). Si querés adelantar algo antes de la clase 10,
que sea el paso 1:

1. **Definí la categoría** por escrito, con casos límite resueltos.
2. **Etiquetá 300 casos al azar**, a ciegas, sin mirar lo que dijo ningún modelo.
3. **Que 60 de esos los codifique otra persona** y calculá el kappa. Si da bajo, parás:
   el problema es el concepto, no el modelo.
4. **Elegí estrategia** según cuántas etiquetas conseguiste (la curva de la sección 8).
5. **Medí el piso** —clase mayoritaria y diccionario— antes de festejar nada.
6. **Evaluá en test**, una sola vez, y **leé 30 errores**.
7. **Reportá**: modelo y versión exactos, cuántas etiquetas y de quién, cómo partiste,
   el piso, las métricas con su frase, la matriz y ejemplos de errores.

**Lo que queda prometido.** Tenemos un número bueno. ¿Eso prueba que medimos lo que
decimos que medimos? **No.** El detector de sátira de la sección 11 tenía un número
bueno. La clase 10 es sobre esa distancia — validez, sesgo, reproducibilidad y el
revisor 2.

---

### Para seguir

`referencias.md`, en esta misma carpeta. Empezá por Barberá et al. (2021): es
prácticamente el manual de esta clase.